# 玻璃棒：结构修复后的清洗结果
本轮只做局部清洗与结构恢复，不判断概念相关性、不联合提炼、无模型调用。**最终正文在第2节，直接展开完整显示。** 本轮没有生成独立HTML页面。

改进：恢复3段被链接密度规则误挡的正文；只剪掉摘要尾部导航；引文URL移到修复记录；从新抓取HTML恢复明确的字段对。百度字段关系仍未恢复，保留原片段，不猜配对。

In [ ]:
from pathlib import Path
import sys,json,html
import pandas as pd
from IPython.display import display, Markdown, HTML
from demiflow.standalone import local_data
ROOT=Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
RUN=ROOT/'state/curation/cleaning_glass_v6'
data=local_data()
# DOC为空显示全部；填文档ID前缀可只看一篇。
DOC=''
documents=data.read_records(RUN/'documents.jsonl').map(lambda r:r['value'])
rows=documents.filter(lambda r:r['doc_id'].startswith(DOC)).take(100)
def full_text(text):
    display(HTML('<pre style="white-space:pre-wrap;overflow-wrap:anywhere;font-size:15px;line-height:1.65">'+html.escape(text)+'</pre>'))

## 1．本轮实际处理结果
“没有保留正文”的原始材料仍在数据表中；这里的数量不代表事实已核验。

In [ ]:
summary=json.loads((RUN/'summary.json').read_text())
display(pd.DataFrame([{'文档':r['title'],'文档ID':r['doc_id'][:8],'修复前字符':len(r['variants']['block_quality']),'修复后字符':len(r['final_text']),'修复动作':len(r['repairs']),'结构未恢复片段':len(r['unresolved_structure'])} for r in rows]))
display(Markdown('原文定位已核对；14项既有内容锚点均保留。模型调用：**0**。'))

## 2．最终保留的清洗正文
完整显示5篇正文，不需要打开其他文件。保留来源文章顺序和列表；字段仍断裂的片段不伪造为表格。这里是清洗材料，不是提炼后的知识。

In [ ]:
for r in rows:
    if not r['final_text']:continue
    display(Markdown('### '+r['title']+' · '+r['doc_id'][:8]))
    display(Markdown('来源：['+r['title']+']('+r['url']+')'))
    if r['unresolved_structure']:display(Markdown('**此篇仍有字段结构未恢复，下文原样保留；第4节列出具体片段。**'))
    full_text(r['final_text'])

## 3．从真实HTML恢复字段关系
只展示仍有保留正文的文档所对应的HTML字段。以下来自**另一次抓取的HTML快照**，用于检验结构恢复；没有回填旧文本、没有替代百度字段。网页交易字段与侧栏候选不混入最终正文。

In [ ]:
dom=data.read_records(RUN/'retained_dom_fields.jsonl').map(lambda r:r['value']).filter(lambda r:r['doc_id'].startswith(DOC)).take(100)
for r in dom:
    if not r['dom_fields']:continue
    display(Markdown('来源：'+r['url']))
    display(pd.DataFrame([{'字段':x['field'],'值':x['value']} for x in r['dom_fields']]))
    display(Markdown('每对字段的XPath及HTML SHA保存在该步原始输出中，可在下一行读取。'))

## 4．还没有解决的字段片段
缺少原始HTML关系时，保留原片段并说明原因。没有凭词义猜测配对，也没有借其他网页的表格覆盖本页。

In [ ]:
for r in rows:
    if not r['unresolved_structure']:continue
    display(Markdown('### '+r['title']))
    display(pd.DataFrame([{'原字段片段':x['text'],'原块ID':x['block_id']} for x in r['unresolved_structure']]))
display(Markdown('百度HTML抓取返回403；字段名和值粘在相邻行中，当前不能可靠还原。'))

## 5．逐项修复：修改前、修改后
原块保留。引文URL在每项edits中，正文只保留引用序号。恢复正常正文时不会顺带恢复参考文献区。

In [ ]:
for r in rows:
    if not r['repairs']:continue
    display(Markdown('### '+r['title']))
    for e in r['repairs']:
        display(Markdown('**'+e['action']+' · '+e['reason']+' · '+e['block_id']+'**'))
        display(Markdown('修改前'));full_text(e['before'])
        display(Markdown('修改后'));full_text(e['after'])

## 6．本轮未保留正文的材料
保持上一轮的清洗准入结果。截断页面需要补取完整正文，不能解释为概念没有知识。

In [ ]:
display(pd.DataFrame([{'文档':r['title'],'原因':r['filter_diagnostics']['document_reason'],'原文字符':len(r['raw_text'])} for r in rows if not r['final_text']]))

## 7．检查真实中间数据
下面保留可直接运行的读取方式；修改DOC后可用以上格重新查看。业务处理为 demiflow 的 `RepairSourceBlocks` 与 `ReadDOMFields`；本notebook只读取已完成结果，不重跑模型。

In [ ]:
# 例如：查看第一篇的真实修复记录，或HTML字段的XPath与来源哈希。
# rows[0]['repairs']
# dom[0]['dom_fields']
# rows[0]['repaired_blocks']
pass